<div style="text-align: center; color:green; font-family: 'Georgia'; font-size: 30px"> <b>FunnyBot</b> </div>

<span style="font-family: 'Georgia'; font-size: 15px;">
    
# <span style="color:green"> Discription: 

Developed an interactive chatbot for casual conversation and entertainment using Python and NLTK. The chatbot identifies user intents by comparing processed input with predefined conversation patterns stored in JSON, then selects an appropriate response.

Applied tokenization, part-of-speech tagging, and WordNet lemmatization to normalize text, alongside punctuation removal and lowercase conversion. Implemented intent matching through shared-word scoring and regular expressions to extract information such as the user’s name and favourite colour. Added session memory and response templates to personalize replies, randomized response selection for variety, and fallback messages for unrecognized input or missing information.</span>
</span>

## <span style="color:green">Technologies:
Python, NLTK, WordNet, regular expressions, JSON. </span> 


In [ ]:
################################################################ loading libraries ###########################################################
import pandas as pd
import random
import nltk
import re
import datetime
from nltk.stem import WordNetLemmatizer
import json

nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('universal_tagset' , quiet=True)
nltk.download('universal_tagset', quiet=True)
nltk.download('punkt_tab' , quiet=True)
nltk.download('wordnet' , quiet=True)
nltk.download('stopwords' , quiet=True)

wnl = WordNetLemmatizer()

################################################################ Customizing stop words ######################################################

stopword_list = nltk.corpus.stopwords.words('english')
stropwords_list = [word for word in stopword_list if word not in ["i", "my", "you", "your", "there", "this", 'is', "how", "can",
                                                                  "what", "do", "who", "not", "get", "why", "it" ]]

###################################################################### memory ###############################################################
memory = {
    "name": "",
    "color": "",
    "time": ""
}

############################################################### cleaning and lemmatizing #####################################################

# takes the nltl universal tag and returns the coerresponding WordNet POS tag
def nltk2wordNet(nltkpos):
    
    if nltkpos.startswith('VERB'):
        return 'v'
    elif (nltkpos.startswith('ADJ')):
        return 'a'
    elif (nltkpos.startswith('ADV')):
        return 'r'
    elif (nltkpos.startswith('NOUN')):
        return 'n'
    else:
        return ""
def remove_regex(text):
    #remove "?i:"
    text = re.sub(r"\?i:", "", text)
    #remove "([A-Z][a-z]+)"
    text = re.sub(r"(\(\[A-Z\]\[a-z\]\+\))", "", text)
    #remove ([A-Za-z]+(?:\\s+[A-Za-z]+){0,2})
    text = re.sub(r"\(\[A-Za-z\]\+\(\?:\\s\+\[A-Za-z\]\+\)\{0,2\}\)", "" , text)
    return text


#takes the sentence and return a list of lemmatised tokens after cleaning them with removing stopwords, punctuations and special chatacters
def lemmatize_text(text):
    
    #remove regex patterns from the text
    text = remove_regex(text)
    
    # tokenize the text
    tokens = nltk.word_tokenize(text)
    
    

    # get the POS tags for each token
    pos_tags = nltk.pos_tag(tokens, tagset='universal')
    
    #remove special characters and punctiations
    cleaned_pos_tags = []
    for token,tag in pos_tags:
        #remove punctuations and special characters
        token = re.sub(r"[^\w\s]", "", token)
        if token != "":
            cleaned_pos_tags.append((token,tag))
   
   
    # map the nltk pos to wordNet pos and lemmatize tokens based on their pos
    lemmatised_tokens =[]
    
    for token, pos in cleaned_pos_tags:
        
        wordnet_pos = nltk2wordNet(pos)
        
        if wordnet_pos != "":
            #if the wordNet pos is vern, noun,adj or adv lemettize it and add to the lemmatized token list
            lemmatised_tokens.append(wnl.lemmatize(token.lower(), wordnet_pos))
        #if not pos is not verb, adj, adv or noun and is not in the stopword list append it to the lemmatized tokens list 
        # as its original, token in stopword_list:
        else:
            lemmatised_tokens.append(token.lower())
    
    return lemmatised_tokens

############################################################# responses and patterns ###########################################################

#creating a dictionaries for responses1 and a dictionary to detect which category responses are based on the memory
def get_int2responses1(intents):
    int2responses1 = {}
    int2placeholders = {}
    for intent in intents:
        int2responses1[intent['tag']] = intent['responses1']
        if intent.get("placeholder"):
            int2placeholders[intent["tag"]] = intent["placeholder"]

    return int2responses1, int2placeholders

#creating a dictionaries for responses2 from a dataFrame called intents
def get_int2responses2(intents):
    int2responses2 = {}
    for intent in intents:
        if intent.get("responses2"):
            int2responses2[intent['tag']] = intent['responses2']
    return int2responses2

#converts the intents patterns of the dataframe intents to the lists of lemmatized tokens
def get_int2lemmatizedPatterns(intents):
    int2lemmatizedPatterns ={}
    for intent in intents:
        tempPattern =[]
        for pattern in intent["patterns"]:
            tempPattern.append(lemmatize_text(pattern))
        int2lemmatizedPatterns[intent["tag"]] = tempPattern
    return int2lemmatizedPatterns

#extract a dictionary of the regex patterns from the intents with pattern templete 
def get_regexPatterns2int(intents):
    regexPatterns2int ={}
    for intent in intents:
        #if the patterns in the intent is  regex
        if intent.get("regexName"):
            regexPattern ="|".join(intent["patterns"])
            regexPatterns2int[regexPattern] = { "tag": intent["tag"], "regexName": intent["regexName"]}
    return regexPatterns2int


############################################################# loading data from a json file###########################################################

#reads the json file. it is a customized read_json to handle exceptions
def read_json(filePath, intentsName):
    try:
        with open (filePath, "r") as file:
            data = json.load(file)
            intents = data[intentsName]
            return intents
    #if reading the json file fails print the error and return an empty dataframe 
    except Exception as error:
        print("loading json file failed!")
        print (error)
        return pd.DataFrame()



#returns int2lemmatizedPatterns, int2responses and regExPattern2int dictionaries and regex from the json file
def load_files(filePath, intentTask):
    #read the json file and convert it to a dataframe
    intents = read_json(filePath, intentTask )
    
    #generate dictionaries for rgx patterns and lemmatized responses and lemmatized patterns
    int2lemmatizedPatterns = get_int2lemmatizedPatterns(intents)
    int2responses1, int2placeholders = get_int2responses1(intents)
    int2responses2 = get_int2responses2(intents)
    regexPatterns2int = get_regexPatterns2int(intents)
      
    return int2lemmatizedPatterns, int2responses1, int2placeholders, int2responses2,regexPatterns2int


int2lemmatizedPatterns, int2responses1, int2placeholders ,int2responses2 ,regexPatterns2int = load_files('intents.json', "intents-task4")

#########################################################detect intent category###########################################################################

def get_matchingScore(tokens1,tokens2):
    set1 = set(tokens1)
    set2 = set(tokens2)
    newSet = set1&set2
    return len(newSet)

#this function takes a re.Match and itterate through its groups and finds the matched group after group(0)
#it is used so we can have different patterns of regex for a tag. For example My name is ([A-Z][a-z]+) and It is ([A-Z][a-z]+) together
def find_matchedGroupIndex(match):
    for i in range(1,len(match.groups())+1): 
        if match.group(i) != None:
            return i
#detects the intent category of the user input and extract the placeholder or memory property value if applicable
def detect_intent_category(userInput):
    #extract the lemmatized tokens from the user input
    lemmatizedUserInput = lemmatize_text(userInput)
   
    #lastScore is used to find the most match between lennetized user input and lemmatized patterns
    lastScore = 0
    matchedCategory = ""
    placeholder = ""
  
    #find the most matching category
    for category, patterns in int2lemmatizedPatterns.items():
        for pattern in patterns:
        
            tempScore = get_matchingScore(lemmatizedUserInput, pattern)
            
            # which score less than 3, the response are more likely irrelevant
            if tempScore >lastScore and tempScore >=1:
                matchedCategory = category
                lastScore = tempScore
               
            #check if the pattern is equal to the user input immidiately return the category
            #it is helpful when the user input are one word
            elif pattern == lemmatizedUserInput:
                matchedCategory = category
             
    #remove extra spaces from user input text
    userInput = re.sub(r"\s+", " ", userInput).strip()
    
    #remove special characters and punctuations from the user input
    userInput = re.sub(r"[^\w\s]", "", userInput)
    
    #if the matched category is existing as a value in regexPatterns2int or if no category is matched, itterate throught the 
    #regexPatterns2int dictionary and check if there is any match. If ther is save the extracted data in the memory and 
    # set the matched category to the category of the regex
    
    #check if the matched category is in the regex patterns
    for(regexPattern, category) in regexPatterns2int.items():
        if(matchedCategory == category["tag"]):
            matched = None
            
            #if the catefory is for taking name do not user re.IGNORECASE
            if category["regexName"] == "name":
                matched= re.search(regexPattern, userInput)
            else:
                matched = re.search(regexPattern, userInput, re.IGNORECASE)
            
            if matched:
                matchedGroupIndex = find_matchedGroupIndex(matched)
                memory[category["regexName"]] = matched.group(matchedGroupIndex)
                break;
            else:

                memory[category["regexName"]]=""
            break

    #check if the catefory responses are takeing data from memory            
    for tag, ph in int2placeholders.items():
        if matchedCategory == tag:
            placeholder = ph
           
    
    return {"tag":matchedCategory, "placeholder":placeholder}

#################################################################################generate response #####################################################

def get_response(userInput):
    #detect the intent category
    detectedCategory = detect_intent_category(userInput)
    
    #updat time in memory
    now= datetime.datetime.now()
    memory["time"] = now.strftime("%H:%M")
    
    #if category is empty
    if detectedCategory["tag"] == "":
        return "I'm sorry, could you rephrase that again"
    
    #if the category has no place holder chice one of the options from the response1 options and return it
    if detectedCategory["placeholder"] == "":
        response = random.choice(int2responses1[detectedCategory["tag"]])
        return response
    
    # if the algorithm could extract the placeholder value from the user input, use the value in one of the 
    # choices in the responses1 and insert the placeholder value in the response
    if detectedCategory["placeholder"] !=  "" and  memory[detectedCategory["placeholder"]] != "" :
        response = random.choice(int2responses1[detectedCategory["tag"]])
        return response.format(**memory)
          
    # if the placeholder value is not found from the user input, use one of the choices in the responses2 
    #without placeholder
    response = random.choice(int2responses2[detectedCategory["tag"]])
    return response

#function takes user input and prints it along with the chatbot response 
def chat():
    # User welcome and termination instruction.
    print(
"""
To exit at any point type quit!

FunnyBot: Welcome! I am a chatBot designed to to help you to pass your time when you're bored!

""", flush=True)
    
    # Main loop:
    while True:
        # 1. Get user input
        user_input = input( "You: ")

        # 2. Check if the user wants to exit
        if user_input== "quit":
            print("FunnyBot: Goodbye!", flush=True)
            break
        # Generate chatbot response
        bot_response = get_response(user_input)
        # to prevent output buffering in the user-chatbot interactions the flush is set to true
        print(f"FunnyBot: {bot_response}", flush=True)     
# Start chatting with the bot
chat()



To exit at any point type quit!

FunnyBot: Welcome! I am a chatBot designed to to help you to pass your time when you're bored!


You: hi
FunnyBot: Hola human, please tell me your FunnyBot user
You: how are you
FunnyBot: Hi, I am great, how are you? Please tell me your FunnyBot user
You: my name is hamed
FunnyBot: what is your name again?
You: my name is Hamed
FunnyBot: OK! Hola Hamed, how can I help you?
You: do you know what is my name
FunnyBot: Your name is Hamed, how can I help you?
You: do you know what is my identity
FunnyBot: Your name is Hamed , how can I help you?
You: i like blue as color
FunnyBot: Sorry your favourite color is ...?
You: my favourite color is blue
FunnyBot: Grate taste| I am color blind though. I do not know what blue looks like!
You: tell me a joke
FunnyBot: Descartes walks into a pub. 'Would you like a beer sir?' asks the landlord politely. Descartes replies, 'I think not' and ping! he vanishes.
You: telling me a joke
FunnyBot: 'Where are you going on holi